In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [5]:
df1 = spark.createDataFrame([(1, 'Robert'), (2, 'Ria'), (3, 'James')], ('emp_id int, emp_name string'))
df2 = spark.createDataFrame([(2, 'USA'), (4, 'India')], ('emp_id int, country string'))

df1.show()
df2.show()

+------+--------+
|emp_id|emp_name|
+------+--------+
|     1|  Robert|
|     2|     Ria|
|     3|   James|
+------+--------+

+------+-------+
|emp_id|country|
+------+-------+
|     2|    USA|
|     4|  India|
+------+-------+



### Inner join (Default)

In [6]:
df1.join(df2, on = df1.emp_id == df2.emp_id).show()

+------+--------+------+-------+
|emp_id|emp_name|emp_id|country|
+------+--------+------+-------+
|     2|     Ria|     2|    USA|
+------+--------+------+-------+



### Left join

In [7]:
df1.join(df2, on = df1.emp_id == df2.emp_id, how = 'left').show()

+------+--------+------+-------+
|emp_id|emp_name|emp_id|country|
+------+--------+------+-------+
|     1|  Robert|  NULL|   NULL|
|     3|   James|  NULL|   NULL|
|     2|     Ria|     2|    USA|
+------+--------+------+-------+



### Right join

In [8]:
df1.join(df2, on = df1.emp_id == df2.emp_id, how = 'right').show()

+------+--------+------+-------+
|emp_id|emp_name|emp_id|country|
+------+--------+------+-------+
|     2|     Ria|     2|    USA|
|  NULL|    NULL|     4|  India|
+------+--------+------+-------+



### Full join

In [9]:
df1.join(df2, on = df1.emp_id == df2.emp_id, how = 'full').show()

+------+--------+------+-------+
|emp_id|emp_name|emp_id|country|
+------+--------+------+-------+
|     1|  Robert|  NULL|   NULL|
|     2|     Ria|     2|    USA|
|     3|   James|  NULL|   NULL|
|  NULL|    NULL|     4|  India|
+------+--------+------+-------+



### Cross join

In [10]:
# Using join function
df1.join(df2, on = None, how = 'cross').show()

+------+--------+------+-------+
|emp_id|emp_name|emp_id|country|
+------+--------+------+-------+
|     1|  Robert|     2|    USA|
|     1|  Robert|     4|  India|
|     2|     Ria|     2|    USA|
|     3|   James|     2|    USA|
|     2|     Ria|     4|  India|
|     3|   James|     4|  India|
+------+--------+------+-------+



In [13]:
# Using crossJoin function
df1.crossJoin(df2).show()

+------+--------+------+-------+
|emp_id|emp_name|emp_id|country|
+------+--------+------+-------+
|     1|  Robert|     2|    USA|
|     1|  Robert|     4|  India|
|     2|     Ria|     2|    USA|
|     3|   James|     2|    USA|
|     2|     Ria|     4|  India|
|     3|   James|     4|  India|
+------+--------+------+-------+



### Left anti join
- Returns rows from the left DataFrame where NO match is found in the right DataFrame
- Only columns from the left DataFrame are included in the result
- SQL equivalent: NOT EXISTS

In [11]:
df1.join(df2, on = df1.emp_id == df2.emp_id, how = 'left_anti').show()

+------+--------+
|emp_id|emp_name|
+------+--------+
|     1|  Robert|
|     3|   James|
+------+--------+



### Left semi join
- Returns rows from the left DataFrame where a match is found in the right DataFrame
- Only columns from the left DataFrame are included in the result
- Similar to an INNER JOIN but only returns left DataFrame columns
- More efficient than INNER JOIN when you only need left columns ( Similar to EXISTS in SQL)


In [12]:
df1.join(df2, on = df1.emp_id == df2.emp_id, how = 'left_semi').show()

+------+--------+
|emp_id|emp_name|
+------+--------+
|     2|     Ria|
+------+--------+



### Self join

In [22]:
# Here joining column is same hence no need of alias
df1.join(df1, on = df1.emp_id == df1.emp_id).show()

+------+--------+------+--------+
|emp_id|emp_name|emp_id|emp_name|
+------+--------+------+--------+
|     1|  Robert|     1|  Robert|
|     2|     Ria|     2|     Ria|
|     3|   James|     3|   James|
+------+--------+------+--------+



In [21]:
# Using alias as joining columns are different
from pyspark.sql.functions import col
df3 = spark.createDataFrame([(1, 'Robert', 2), (2, 'Ria', 3), (3, 'James', 5)], ('emp_id int, emp_name string, manager_id int'))
df3.show()

df3.alias('emp').join(df3.alias('mgr'), col('mgr.emp_id') == col('emp.manager_id')).select(col('emp.emp_id'), col('emp.emp_name'), col('mgr.emp_name').alias('manager_name')).show()

+------+--------+----------+
|emp_id|emp_name|manager_id|
+------+--------+----------+
|     1|  Robert|         2|
|     2|     Ria|         3|
|     3|   James|         5|
+------+--------+----------+

+------+--------+------------+
|emp_id|emp_name|manager_name|
+------+--------+------------+
|     1|  Robert|         Ria|
|     2|     Ria|       James|
+------+--------+------------+



### Multi-column join

In [29]:
emp = spark.createDataFrame([(1, 101, 'Robert'), (2, 102, 'Ria'), (3, 103, 'James')], ('emp_id int, dep_id int, emp_name string'))
country = spark.createDataFrame([(2, 102, 'USA'), (4, 104, 'India')], ('emp_id int, dep_id int, country string'))
hire = spark.createDataFrame([(1, '01-Jan-2021'), (2, '01-Feb-2021'), (3, '01-Mar-2021')], ('emp_id int, hire_date string'))

emp.show()
country.show()
hire.show()


+------+------+--------+
|emp_id|dep_id|emp_name|
+------+------+--------+
|     1|   101|  Robert|
|     2|   102|     Ria|
|     3|   103|   James|
+------+------+--------+

+------+------+-------+
|emp_id|dep_id|country|
+------+------+-------+
|     2|   102|    USA|
|     4|   104|  India|
+------+------+-------+

+------+-----------+
|emp_id|  hire_date|
+------+-----------+
|     1|01-Jan-2021|
|     2|01-Feb-2021|
|     3|01-Mar-2021|
+------+-----------+



In [27]:
emp.join(country, (emp.emp_id == country.emp_id) & (emp.dep_id == country.dep_id)).show()

+------+------+--------+------+------+-------+
|emp_id|dep_id|emp_name|emp_id|dep_id|country|
+------+------+--------+------+------+-------+
|     2|   102|     Ria|     2|   102|    USA|
+------+------+--------+------+------+-------+



### Multiple Dataframe Join

In [30]:
emp.join(country, (emp.emp_id == country.emp_id) & (emp.dep_id == country.dep_id)).join(hire, emp.emp_id == hire.emp_id).show()

+------+------+--------+------+------+-------+------+-----------+
|emp_id|dep_id|emp_name|emp_id|dep_id|country|emp_id|  hire_date|
+------+------+--------+------+------+-------+------+-----------+
|     2|   102|     Ria|     2|   102|    USA|     2|01-Feb-2021|
+------+------+--------+------+------+-------+------+-----------+

